In [1]:
# run it for the first time
# %pip install hf_xet

In [1]:
import json
from pathlib import Path
from typing import List
from tqdm import tqdm

import requests

In [2]:
SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/questions/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/questions/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/questions/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/questions/rpm_sa_questions.json"),
}

SELECTED_DATASETS = ["DoS", "Fuzzy", "Gear", "RPM"]


In [7]:
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
MODEL_ID = "gemma2:9b"
MODEL_TAG = MODEL_ID.replace(":", "_").replace("-", "_")

print(f"Ollama URL: {OLLAMA_BASE_URL}")
print(f"Model: {MODEL_ID}")

Ollama URL: http://127.0.0.1:11434
Model: gemma2:9b


In [8]:
import re

MAX_WORDS = 10

def build_explanation_prompt(context: str, question: str, ground_truth: str) -> str:
    """Generate prompt for explaining why a ground_truth answer is correct."""
    system_prompt = (
        "You are a CAN-bus anomaly analysis expert. "
        "You are given a CAN time-window log, a question, and the CORRECT answer. "
        "Generate a very short explanation (max 10 words) of WHY that answer is correct "
        "by analyzing evidence in the CAN log. "
        "Focus on specific patterns: timestamp gaps, ID frequencies, payload values, DLC changes, "
        "Flag patterns, or baseline behavior. Be specific and cite evidence."
    )

    return (
        f"{system_prompt}\n\n"
        f"CAN log:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Correct Answer:\n{ground_truth}\n\n"
        "Provide a very short explanation (max 10 words) of why this answer is correct based on the CAN log:"
    )


def _extract_ollama_text(response_json: dict) -> str:
    if isinstance(response_json, dict):
        if isinstance(response_json.get("response"), str):
            return response_json["response"]
        if "message" in response_json and isinstance(response_json["message"], dict):
            content = response_json["message"].get("content")
            if isinstance(content, str):
                return content
    return ""


def _limit_words(text: str, max_words: int = MAX_WORDS) -> str:
    words = text.strip().split()
    return " ".join(words[:max_words])


def generate_explanation(context: str, question: str, ground_truth: str, max_tokens: int = 60) -> str:
    """Generate LLM explanation for why ground_truth is correct given context."""
    prompt = build_explanation_prompt(context, question, ground_truth)
    
    payload = {
        "model": MODEL_ID,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.3,
            "num_predict": max_tokens,
        },
    }
    
    try:
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=payload,
            timeout=180,
        )
        resp.raise_for_status()
        completion = _extract_ollama_text(resp.json())
        return _limit_words(completion)
    except Exception as e:
        print(f"[ERROR] Explanation generation failed: {e}")
        return ""


def load_questions(path) -> list:
    """Load questions from .json (list) or .jsonl."""
    from pathlib import Path
    import json
    
    path = Path(path)
    if not path.exists():
        return []
    
    if path.suffix == ".json":
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

In [ ]:
# # Quick sanity check: generate explanations for a small sample
# SAMPLE_PER_DATASET = 2
# SAMPLE_SUFFIX = f"explanations_sample_{SAMPLE_PER_DATASET}"

# for ds_name in SELECTED_DATASETS:
#     q_path = SA_QUESTION_FILES[ds_name]
#     if not q_path.exists():
#         print(f"[WARN] Questions for {ds_name} not found at {q_path}, skip.")
#         continue

#     questions = load_questions(q_path)
#     if not questions:
#         print(f"[WARN] {ds_name}: no questions loaded, skip.")
#         continue

#     sample_questions = questions[:SAMPLE_PER_DATASET]
#     print(f"[INFO] {ds_name}: generating explanations for {len(sample_questions)} samples.")

#     out_dir = q_path.parent.parent / "llm_explanations"
#     out_dir.mkdir(parents=True, exist_ok=True)
#     ans_path = out_dir / f"{ds_name.lower()}_sa_answers_{MODEL_TAG}_{SAMPLE_SUFFIX}.jsonl"

#     ans_path.write_text("", encoding="utf-8")

#     with ans_path.open("a", encoding="utf-8") as f:
#         for rec in tqdm(sample_questions, desc=f"{ds_name} explanations"):
#             qa_id = rec.get("qa_id")
#             metadata = rec.get("metadata", {})
#             dataset = metadata.get("dataset", ds_name)
#             context = rec["context"]
#             question = rec["question"]
#             gt = rec.get("ground_truth", rec.get("answer", ""))

#             # Generate explanation for the ground_truth
#             explanation = generate_explanation(context, question, gt)
#             if not explanation:
#                 print(f"[WARN] Empty explanation: dataset={ds_name}, qa_id={qa_id}")

#             answer_rec = {
#                 "qa_id": qa_id,
#                 "dataset": dataset,
#                 "sa_type": rec.get("sa_type"),
#                 "model": MODEL_ID,
#                 "llm_answer": explanation,  # Explanation of ground_truth
#                 "ground_truth": gt,
#             }
#             f.write(json.dumps(answer_rec, ensure_ascii=False) + "\n")

#     print(f"[INFO] {ds_name}: sample explanations saved to {ans_path}")

In [10]:
# Full generation: Generate explanations for ALL questions in each dataset
for ds_name in SELECTED_DATASETS:
    q_path = SA_QUESTION_FILES[ds_name]
    if not q_path.exists():
        print(f"[WARN] Questions for {ds_name} not found at {q_path}, skip.")
        continue

    questions = load_questions(q_path)
    print(f"[INFO] {ds_name}: loaded {len(questions)} questions.")

    out_dir = q_path.parent.parent / "llm_explanations"
    out_dir.mkdir(parents=True, exist_ok=True)
    ans_path = out_dir / f"{ds_name.lower()}_sa_explanations_{MODEL_TAG}.jsonl"

    ans_path.write_text("", encoding="utf-8")

    with ans_path.open("a", encoding="utf-8") as f:
        for rec in tqdm(questions, desc=f"{ds_name} explanations"):
            qa_id = rec.get("qa_id")
            metadata = rec.get("metadata", {})
            dataset = metadata.get("dataset", ds_name)
            context = rec["context"]
            question = rec["question"]
            gt = rec.get("ground_truth", rec.get("answer", ""))

            # Generate explanation for the ground_truth
            explanation = generate_explanation(context, question, gt)
            if not explanation:
                print(f"[WARN] Empty explanation: dataset={ds_name}, qa_id={qa_id}")

            answer_rec = {
                "qa_id": qa_id,
                "dataset": dataset,
                "sa_type": rec.get("sa_type"),
                "model": MODEL_ID,
                "llm_answer": explanation,  # Explanation of why ground_truth is correct
                "ground_truth": gt,
            }
            f.write(json.dumps(answer_rec, ensure_ascii=False) + "\n")

    print(f"[INFO] {ds_name}: explanations saved to {ans_path}")

[INFO] DoS: loaded 916 questions.


DoS explanations: 100%|██████████| 916/916 [1:45:02<00:00,  6.88s/it]


[INFO] DoS: explanations saved to DoS_sa_qa\llm_explanations\dos_sa_explanations_gemma2_9b.jsonl
[INFO] Fuzzy: loaded 959 questions.


Fuzzy explanations: 100%|██████████| 959/959 [2:01:42<00:00,  7.61s/it]  


[INFO] Fuzzy: explanations saved to Fuzzy_sa_qa\llm_explanations\fuzzy_sa_explanations_gemma2_9b.jsonl
[INFO] Gear: loaded 1110 questions.


Gear explanations: 100%|██████████| 1110/1110 [2:17:19<00:00,  7.42s/it] 


[INFO] Gear: explanations saved to Gear_sa_qa\llm_explanations\gear_sa_explanations_gemma2_9b.jsonl
[INFO] RPM: loaded 1155 questions.


RPM explanations: 100%|██████████| 1155/1155 [2:29:51<00:00,  7.79s/it] 

[INFO] RPM: explanations saved to RPM_sa_qa\llm_explanations\rpm_sa_explanations_gemma2_9b.jsonl
